In [2]:
%pip install pypdf langchain-community langchain-text-splitters chromadb langchain-google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\PC\AppData\Local\Temp\ipykernel_13976\2489923474.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
pdf_folder = Path(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\documents")

pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

Number of PDFs: 0


In [10]:
print("Folder exists:", pdf_folder.exists())
print("Files inside the folder:")

for file in pdf_folder.iterdir():
    print(file.name)

Folder exists: False
Files inside the folder:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\PC\\Desktop\\100 DAYS OF AI\\WEEK 3\\DAY 21\\documents'

In [11]:
from pathlib import Path
import shutil

day21_folder = Path(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21")
pdf_folder = day21_folder / "documents"

# Create the documents folder
pdf_folder.mkdir(exist_ok=True)

# Location of your current PDF
source_pdf = Path(r"C:\Users\PC\Documents\karthika journal.pdf")

# Copy the PDF into our Day 21 documents folder
shutil.copy2(source_pdf, pdf_folder / "karthika journal.pdf")

print("PDF copied successfully!")
print("Location:", pdf_folder / "karthika journal.pdf")

PDF copied successfully!
Location: C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\documents\karthika journal.pdf


In [12]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print(pdf.name)

Number of PDFs: 1
karthika journal.pdf


In [13]:
all_pages = []

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    pages = loader.load()
    all_pages.extend(pages)

print("Total pages loaded:", len(all_pages))

Total pages loaded: 5


In [14]:
all_pages = []

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    pages = loader.load()
    all_pages.extend(pages)

print("Total pages loaded:", len(all_pages))

Total pages loaded: 5


In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_pages)

print("Number of chunks:", len(chunks))

Number of chunks: 48


In [16]:
load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env")

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Embedding model ready!")

Embedding model ready!


In [17]:
chunk_texts = [chunk.page_content for chunk in chunks]

chunk_vectors = embeddings.embed_documents(chunk_texts)

print("Number of vectors:", len(chunk_vectors))
print("Vector dimensions:", len(chunk_vectors[0]))

Number of vectors: 48
Vector dimensions: 3072


In [18]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="rag_documents"
)

print("Chroma database ready!")

Chroma database ready!


In [19]:
metadatas = [
    {
        "source": Path(chunk.metadata["source"]).name,
        "page": chunk.metadata.get("page", 0) + 1
    }
    for chunk in chunks
]

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunk_texts,
    embeddings=chunk_vectors,
    metadatas=metadatas
)

print("Chunks stored in Chroma!")

Chunks stored in Chroma!


In [20]:
query = "What are the three severity levels?"

query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

print("Retrieved chunks:\n")

for i, doc in enumerate(results["documents"][0]):
    print(f"--- Chunk {i + 1} ---")
    print(doc)
    print()

Retrieved chunks:

--- Chunk 1 ---
moderate, indicating more significant damage covering a larger 
surface area or affecting panel alignment; and severe, 
representing extensive structural damage requiring major repair 
intervention. The severity assignment assists users and 
insurance assessors in making informed decisions regarding 
repair prioritization and claim processing.   
E.  Database and Output Module  
Detection results including bounding box coordinates, 
damage categories, confidence scores , and severity

--- Chunk 2 ---
separation or missing vehicle parts. 
D.  Severity Classification Module  
Following damage detection, the severity classification 
module analyses the spatial extent, depth characteristics, and 
distribution of each identified damage region to assign a 
severity rating. Damage instances are categorized into three 
levels: minor, referring to superficial surface marks with 
limited extent th at do not affect vehicle structural integrity;

--- Chunk 3 ---


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini chatbot ready!")

Gemini chatbot ready!


In [22]:
context = "\n\n".join(results["documents"][0])

prompt = f"""
You are a helpful document chatbot.

Answer the user's question using ONLY the information in the context below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find the answer in the provided documents."

Answer clearly and concisely.
"""

response = llm.invoke(prompt)

answer = response.content[0]["text"]

print("Answer:")
print(answer)

Answer:
Based on the provided context, the three severity levels are:

* **Minor**
* **Moderate**
* **Severe**


In [23]:
def ask_question(question):
    query_vector = embeddings.embed_query(question)

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=3
    )

    context = "\n\n".join(results["documents"][0])

    prompt = f"""
    You are a helpful document chatbot.

    Answer the question using ONLY the information in the context.

    Context:
    {context}

    Question:
    {question}

    If the answer is not present in the context, say:
    "I could not find the answer in the provided documents."

    Answer clearly and concisely.
    """

    response = llm.invoke(prompt)

    return response.content[0]["text"]

In [24]:
print(ask_question("What are the four primary vehicle damage categories?"))

The four primary vehicle damage categories are:

1. **Scratches**
2. **Dents**
3. **Cracks**
4. **Broken components** (or broken vehicle components)


In [26]:
print(ask_question("Whos the author?"))

The authors are M. Vasuki and M. Karthika.


In [27]:
print(ask_question("What is the model performance?"))

Based on the provided context, the model's performance includes:

* **Overall mAP:** 88.7% across four damage categories (at an IoU threshold of 0.50).
* **Severity Estimation Accuracy:** 91.3% overall classification accuracy.
* **Inference Time:** 15 milliseconds per image (the lowest among evaluated methods).
* **Baseline Comparison:** Outperforms all baseline methods across all evaluation metrics.
* **Category Highlights:** Broken part detection recorded the highest accuracy, while scratch detection presented the greatest difficulty due to low-contrast surface characteristics.


In [28]:
print(ask_question("What is the capital of France?"))

I could not find the answer in the provided documents.


In [29]:
def ask_question(question):
    query_vector = embeddings.embed_query(question)

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=3
    )

    context_parts = []

    for i in range(len(results["documents"][0])):
        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        context_parts.append(
            f"Source: {metadata['source']}, Page: {metadata['page']}\n"
            f"{document}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
    You are a helpful document chatbot.

    Answer the question using ONLY the information in the context.

    Context:
    {context}

    Question:
    {question}

    If the answer is not present in the context, say:
    "I could not find the answer in the provided documents."

    Answer clearly and concisely.
    """

    response = llm.invoke(prompt)

    answer = response.content[0]["text"]

    return answer

In [30]:
print(ask_question("What are the three severity levels?"))

The three severity levels are minor, moderate, and severe.


In [31]:
question = "What are the three severity levels?"

query_vector = embeddings.embed_query(question)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

print("Answer:")
print(ask_question(question))

print("\nSources:")

for metadata in results["metadatas"][0]:
    print(f"- {metadata['source']} | Page {metadata['page']}")

Answer:
Based on the provided context, the three severity levels are:

* **Minor**: Superficial surface marks with limited extent that do not affect structural integrity.
* **Moderate**: Significant damage covering a larger surface area or affecting panel alignment.
* **Severe**: Extensive structural damage requiring major repair intervention.

Sources:
- karthika journal.pdf | Page 3
- karthika journal.pdf | Page 3
- karthika journal.pdf | Page 4


In [32]:
while True:
    question = input("\nAsk a question (or type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Chatbot ended.")
        break

    print("\nAnswer:")
    print(ask_question(question))


Ask a question (or type 'exit' to stop):  What is YOLOv10?



Answer:
Based on the provided context, **YOLOv10** is the latest iteration of the YOLO family of real-time object detectors. Key details include:

* **Features & Improvements:** It incorporates dual-assignment training strategies, architectural refinements, and reduced architectural redundancy. These changes improve feature extraction depth, bounding box precision, accuracy, and computational efficiency under constrained computational budgets.
* **Functionality:** In the detection module, it performs end-to-end damage localization and classification by processing images through a backbone feature extractor to generate multi-scale feature representations.



Ask a question (or type 'exit' to stop):  limitation of this journal



Answer:
I could not find the answer in the provided documents.



Ask a question (or type 'exit' to stop):  exit


Chatbot ended.


In [1]:
def chat():
    while True:
        question = input("\nAsk a question (or type 'exit' to stop): ")

        if question.lower() == "exit":
            print("Chatbot ended.")
            break

        query_vector = embeddings.embed_query(question)

        results = collection.query(
            query_embeddings=[query_vector],
            n_results=3
        )

        context_parts = []

        for i in range(len(results["documents"][0])):
            document = results["documents"][0][i]
            metadata = results["metadatas"][0][i]

            context_parts.append(
                f"Source: {metadata['source']}, Page: {metadata['page']}\n"
                f"{document}"
            )

        context = "\n\n".join(context_parts)

        prompt = f"""
        You are a helpful document chatbot.

        Answer the question using ONLY the information in the context.

        Context:
        {context}

        Question:
        {question}

        If the answer is not present in the context, say:
        "I could not find the answer in the provided documents."

        Answer clearly and concisely.
        """

        response = llm.invoke(prompt)
        answer = response.content[0]["text"]

        print("\nAnswer:")
        print(answer)

        print("\nSources:")
        for metadata in results["metadatas"][0]:
            print(f"- {metadata['source']} | Page {metadata['page']}")

In [2]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

NameError: name 'pdf_folder' is not defined

In [3]:
from pathlib import Path

pdf_folder = Path(
    r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\documents"
)

print("Folder exists:", pdf_folder.exists())

Folder exists: True


In [4]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- karthika journal.pdf


In [6]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- karthika journal.pdf


In [7]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- karthika journal.pdf


In [8]:
day21_folder = Path(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21")

pdf_files = list(day21_folder.rglob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

Number of PDFs: 3
C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\class-5-english-marigold-chapter-1-c2d2814f.pdf
C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\karthika journal.pdf
C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\documents\karthika journal.pdf


In [9]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- karthika journal.pdf


In [10]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- karthika journal.pdf


In [11]:
all_pages = []

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    pages = loader.load()
    all_pages.extend(pages)

print("Total pages loaded:", len(all_pages))

NameError: name 'PyPDFLoader' is not defined

In [12]:
from langchain_community.document_loaders import PyPDFLoader

print("PyPDFLoader ready!")

C:\Users\PC\AppData\Local\Temp\ipykernel_23612\1774167615.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PyPDFLoader ready!


In [13]:
all_pages = []

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    pages = loader.load()
    all_pages.extend(pages)

print("Total pages loaded:", len(all_pages))

Total pages loaded: 5


In [14]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- karthika journal.pdf


In [15]:
import shutil
from pathlib import Path

source_pdf = Path(
    r"C:\Users\PC\Documents\class-5-english-marigold-chapter-1-c2d2814f.pdf"
)

destination_pdf = pdf_folder / source_pdf.name

shutil.copy2(source_pdf, destination_pdf)

print("Marigold PDF copied!")
print(destination_pdf)

Marigold PDF copied!
C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\documents\class-5-english-marigold-chapter-1-c2d2814f.pdf


In [16]:
old_pdf = pdf_folder / "karthika journal.pdf"

if old_pdf.exists():
    old_pdf.unlink()
    print("Old PDF removed!")
else:
    print("Old PDF not found.")

Old PDF removed!


In [17]:
pdf_files = list(pdf_folder.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 1
- class-5-english-marigold-chapter-1-c2d2814f.pdf


In [ ]:
all_pages = []

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    pages = loader.load()
    all_pages.extend(pages)

print("Total pages loaded:", len(all_pages))

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_pages)

print("Number of chunks:", len(chunks))

In [20]:
from pypdf import PdfReader

pdf_path = pdf_files[0]

reader = PdfReader(str(pdf_path))

print("Total pages:", len(reader.pages))
print("First page text:")
print(reader.pages[0].extract_text()[:1000])

Total pages: 32
First page text:
Unit□1
What is cold, sweet and creamy, and
wonderful to eat? Everyone's favourite treat especially
on a hot summer day is an ice cream! And everyone's
favourite person might just be the Ice-cream Man!
Ice cream□Man-
Read and Enjoy
2 3
2021-22



In [21]:
from pypdf import PdfReader
from langchain_core.documents import Document

reader = PdfReader(str(pdf_files[0]))

all_pages = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text:
        all_pages.append(
            Document(
                page_content=text,
                metadata={
                    "source": pdf_files[0].name,
                    "page": page_number + 1
                }
            )
        )

print("Pages extracted:", len(all_pages))

Pages extracted: 32


In [22]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_pages)

print("Number of chunks:", len(chunks))

NameError: name 'RecursiveCharacterTextSplitter' is not defined

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Text splitter ready!")


Text splitter ready!


In [24]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_pages)

print("Number of chunks:", len(chunks))

Number of chunks: 78


In [25]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv(
    r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env"
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Embedding model ready!")

Embedding model ready!


In [26]:
chunk_texts = [chunk.page_content for chunk in chunks]

chunk_vectors = embeddings.embed_documents(chunk_texts)

print("Number of vectors:", len(chunk_vectors))
print("Vector dimensions:", len(chunk_vectors[0]))

Number of vectors: 78
Vector dimensions: 3072


In [27]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection("rag_documents")
    print("Old collection deleted.")
except:
    print("No old collection found.")

collection = client.create_collection(
    name="rag_documents"
)

print("Fresh Chroma collection created!")

Old collection deleted.
Fresh Chroma collection created!


In [30]:
chunk_texts = [chunk.page_content for chunk in chunks]

metadatas = [
    {
        "source": chunk.metadata["source"],
        "page": chunk.metadata["page"]
    }
    for chunk in chunks
]

collection.add(
    ids=[f"marigold_chunk_{i}" for i in range(len(chunks))],
    documents=chunk_texts,
    embeddings=chunk_vectors,
    metadatas=metadatas
)

print("Chunks stored in Chroma:", len(chunks))

Chunks stored in Chroma: 78


In [31]:
query = "What is the main story in this chapter?"

query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

print("Retrieved chunks:\n")

for i, doc in enumerate(results["documents"][0]):
    print(f"--- Chunk {i + 1} ---")
    print(doc)
    print()

Retrieved chunks:

--- Chunk 1 ---
Say Aloud
Let's Write
:
:
:
Recycling waste
Folk tales
Multi-cultural approachto
food
To avoid wastage of food
In every country of the world, there are stories which have been handed down from
grandparents to grandchildren or which have been sung by mothers to their babies.
These stories are called and tell us about the customs and culture of the place
they are set in.
A Kerala folk tale and a Santhal folk tale have been retold in this unit. The teacher

--- Chunk 2 ---
all came from a basket of waste!
avial
Avial
Waste can be quite useful !
Find out for yourself from this
story...
Marigold
10 11
2021-22

--- Chunk 3 ---
What are some of the things your parents and
teachers tell you to do on
time? ...Get up in the
morning… do the homework…. pack your
school bag...
What happened when you didn't do as they asked
you to do?
Deep in a forest
stood a very tall tree. Its leafy
branches spread out like strong
arms.
This tree was the home of a flock of wild g

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

context = "\n\n".join(results["documents"][0])

prompt = f"""
You are a helpful document chatbot.

Answer the question using ONLY the information in the context below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find the answer in the provided documents."

Answer clearly and concisely.
"""

response = llm.invoke(prompt)

print("Answer:")
print(response.content[0]["text"])

Answer:
I could not find the answer in the provided documents.


In [33]:
def chat():
    while True:
        question = input("\nAsk a question (or type 'exit' to stop): ")

        if question.lower() == "exit":
            print("Chatbot ended.")
            break

        query_vector = embeddings.embed_query(question)

        results = collection.query(
            query_embeddings=[query_vector],
            n_results=3
        )

        context_parts = []

        for i in range(len(results["documents"][0])):
            document = results["documents"][0][i]
            metadata = results["metadatas"][0][i]

            context_parts.append(
                f"Source: {metadata['source']}, Page: {metadata['page']}\n"
                f"{document}"
            )

        context = "\n\n".join(context_parts)

        prompt = f"""
        You are a helpful document chatbot.

        Answer the question using ONLY the information in the context.

        Context:
        {context}

        Question:
        {question}

        If the answer is not present in the context, say:
        "I could not find the answer in the provided documents."

        Answer clearly and concisely.
        """

        response = llm.invoke(prompt)
        answer = response.content[0]["text"]

        print("\nAnswer:")
        print(answer)

        print("\nSources:")
        for metadata in results["metadatas"][0]:
            print(f"- {metadata['source']} | Page {metadata['page']}")

In [34]:
chat()


Ask a question (or type 'exit' to stop):  What is the story about?



Answer:
Based on the provided context, there are excerpts from a few different stories:

1. **Waste and Avial (Page 10):** A story about how waste can be quite useful (referencing "avial").
2. **Wild Geese (Page 20):** A story about a flock of wild geese living safely in a tall tree in a forest, where a wise old bird notices a small creeper at the foot of the tree.
3. **The Ant and the Dove (Page 25):** A paragraph about an ant that falls into a fountain and is saved from drowning by a friendly dove who drops a leaf for it.

Sources:
- class-5-english-marigold-chapter-1-c2d2814f.pdf | Page 20
- class-5-english-marigold-chapter-1-c2d2814f.pdf | Page 10
- class-5-english-marigold-chapter-1-c2d2814f.pdf | Page 25



Ask a question (or type 'exit' to stop):  What does the chapter teach us?



Answer:
I could not find the answer in the provided documents.

Sources:
- class-5-english-marigold-chapter-1-c2d2814f.pdf | Page 18
- class-5-english-marigold-chapter-1-c2d2814f.pdf | Page 31
- class-5-english-marigold-chapter-1-c2d2814f.pdf | Page 32



Ask a question (or type 'exit' to stop):  exit


Chatbot ended.


In [35]:
print("PDFs:", len(pdf_files))
print("Chunks in Chroma:", collection.count())
print("Chatbot ready: Yes")

PDFs: 1
Chunks in Chroma: 78
Chatbot ready: Yes


# Conclusion

Today I built a basic RAG chatbot that can answer questions using information from PDF documents.

The project loads PDF documents, extracts their text, splits the text into smaller chunks, generates embeddings using Gemini, and stores the chunks and embeddings in Chroma. When a user asks a question, the chatbot retrieves the most relevant chunks and passes them to Gemini to generate an answer.

I also added metadata containing the source document and page number, allowing the chatbot to show where the retrieved information came from.

### RAG Pipeline

PDF → Text Extraction → Chunking → Gemini Embeddings → Chroma → Retrieval → Gemini → Answer

### Key Takeaways

- RAG allows an LLM to answer questions using external documents.
- Chunking makes large documents easier to retrieve from.
- Embeddings allow semantic similarity search.
- Chroma stores and retrieves the document embeddings.
- Metadata helps identify the source and page of retrieved information.
- The same pipeline can be extended to multiple PDF documents.
- A RAG chatbot is more reliable when it is instructed to answer only from retrieved context.

**Day 21 Complete! 🚀**

# Day 21 — RAG Chatbot on Documents

## Project Overview

Today I built a basic RAG (Retrieval-Augmented Generation) chatbot that answers questions using information from PDF documents.

The project uses Gemini for embeddings and answer generation, and Chroma as the vector database.

## What I Built

The chatbot can:

- Load PDF documents
- Extract text from PDF pages
- Split documents into smaller chunks
- Generate Gemini embeddings
- Store embeddings and document text in Chroma
- Retrieve relevant chunks for a user question
- Generate answers using Gemini
- Show the source PDF and page number

## RAG Pipeline

PDF → Text Extraction → Chunking → Gemini Embeddings → Chroma → Retrieval → Gemini → Answer

## Technologies Used

- Python
- PyPDF
- LangChain
- Gemini
- ChromaDB
- Jupyter Notebook

## Current Document

The project was tested using a Class 5 English Marigold PDF.

The document was split into **78 chunks** and stored in the Chroma vector database.

## Key Learnings

- How to build a basic RAG pipeline
- How embeddings enable semantic search
- How Chroma stores and retrieves document embeddings
- How metadata can track document sources and pages
- How retrieved context can be passed to an LLM
- How to build an interactive document chatbot

## Future Improvement

The chatbot can be extended to work with **15–20 PDF documents** and retrieve information across multiple documents.

**Day 21 Complete! 🚀**

In [37]:
from pathlib import Path

readme_lines = [
    "# Day 21 — RAG Chatbot on Documents",
    "",
    "## Project Overview",
    "",
    "Today I built a basic RAG (Retrieval-Augmented Generation) chatbot that answers questions using information from PDF documents.",
    "",
    "The project uses Gemini for embeddings and answer generation, and Chroma as the vector database.",
    "",
    "## What I Built",
    "",
    "The chatbot can:",
    "",
    "- Load PDF documents",
    "- Extract text from PDF pages",
    "- Split documents into smaller chunks",
    "- Generate Gemini embeddings",
    "- Store embeddings and document text in Chroma",
    "- Retrieve relevant chunks for a user question",
    "- Generate answers using Gemini",
    "- Show the source PDF and page number",
    "",
    "## RAG Pipeline",
    "",
    "PDF → Text Extraction → Chunking → Gemini Embeddings → Chroma → Retrieval → Gemini → Answer",
    "",
    "## Technologies Used",
    "",
    "- Python",
    "- PyPDF",
    "- LangChain",
    "- Gemini",
    "- ChromaDB",
    "- Jupyter Notebook",
    "",
    "## Current Document",
    "",
    "The project was tested using a Class 5 English Marigold PDF.",
    "",
    "The document contains 32 pages and was split into 78 chunks.",
    "",
    "## Key Learnings",
    "",
    "- How to build a basic RAG pipeline",
    "- How embeddings enable semantic search",
    "- How Chroma stores and retrieves document embeddings",
    "- How metadata can track document sources and pages",
    "- How retrieved context can be passed to an LLM",
    "- How to build an interactive document chatbot",
    "",
    "## Future Improvement",
    "",
    "The chatbot can be extended to work with 15–20 PDF documents and retrieve information across multiple documents.",
    "",
    "**Day 21 Complete! 🚀**"
]

readme_path = Path(
    r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 3\DAY 21\README.md"
)

readme_path.write_text(
    "\n".join(readme_lines),
    encoding="utf-8"
)

print("README.md created successfully!")

README.md created successfully!
